In [ ]:
# COMPARISON OF NNs WITH PHYSICS-BASED LOTT & MILLER (1997) SCHEME

# LIBRARIESimport math

import sys
import os
import math

import numpy as np
import torch

import datetime as dt
import time

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import SymLogNorm
from matplotlib.colors import Normalize

import colormaps as cmaps 

from _utils import lott1997_1cell as L97

utils_path = os.path.abspath('_utils')

if utils_path not in sys.path:
    sys.path.append(utils_path)

In [ ]:
# INPUT SETTINGS

t_start = dt.datetime( 2022,  7, 1,  0)
t_end = dt.datetime(   2022,  7, 1, 23)
grid = 'n32'
igtype = 'SGIG16' #change
month=dt.datetime.strftime(t_start,'%Y-%m')

t_delta = dt.timedelta(hours=1)
ts = []  # ts in datetime format
ts.append(t_start)
while ts[-1] < t_end:
    ts.append(ts[-1] + t_delta)
steps = len(ts)

path  = '/p/scratch/icon-a-ml/haslauer1/_data/_ERA5/_ML_new/_' + grid + '/' + igtype + '/'
files = []

for i in range(len(ts)):
    month=dt.datetime.strftime(ts[i],'%Y-%m')
    files.append(path + '__' + month + '/box_GLOBAL_70/alt/' + ts[i].strftime('%Y%m%d%H%M%S') + '_zll.npz')

data = []
for i in range(steps):
    data.append(np.load(files[i]))

stats = np.load(path + '/_preprocessed_full2024_land_orostd_/_statistics.npz')
mean=stats['feat_mean']
mean.shape

In [ ]:
# SETUP FOR LOTT & MILLER

omega = 7.2921e-5  # Earth's angular velocity (rad/s)
pcoriolis = 2*omega*np.sin(np.radians(data[0]['lat'][:,0]))


In [ ]:
# ERA5 TIME STEP AND LAYER PREPARATION

pdtime_1h   = 3600
pdtime_24h  = 86400        # seconds per day

papm1 = np.array([100.0, 200.0, 300.0, 500.0, 700.0, 
        1000.0, 2000.0, 3000.0, 5000.0, 7000.0, 
        10000.0, 12500.0, 15000.0, 17500.0, 20000.0, 22500.0, 25000.0,
        30000.0, 35000.0, 40000.0, 45000.0, 50000.0, 55000.0, 60000.0, 65000.0,
        70000.0, 75000.0, 77500.0, 80000.0, 82500.0, 85000.0, 87500.0, 90000.0, 92500.0, 95000.0, 97500.0, 100000.0])

paphm1 = np.zeros(len(papm1)+1)

paphm1[0] = 10
for i in range(1,len(papm1),1):
    paphm1[i] = math.exp((math.log(papm1[i-1]) + math.log(papm1[i]))/2)
paphm1[-1] = 101300.0

In [ ]:
# FUNCTIONS FOR COMPUTING DRAG FROM FLUX AND SCALING

def drag_from_flux(flux_vec, geop_vec, p_vec, ta_vec):
    
    GWD = np.gradient(flux_vec, p_vec) * 9.81
    return GWD
    

def scale_fluxes(flux_vec):
    return (flux_vec - stats['targ_mean']) / stats['targ_std']


def scale_features(feat_vec):
    length = mean.shape[0]
    temp = feat_vec.reshape(-1, length)
    
    mean_matrix = np.stack([stats['feat_mean']]*temp.shape[0], axis=0)
    std_matrix = np.stack([stats['feat_std']]*temp.shape[0], axis=0)
    feat_normalized = ((temp-mean_matrix)/std_matrix).reshape(50, 128, 153)

    return feat_normalized


def rescale_fluxes(flux_vec):
    return flux_vec * stats['targ_std'] + stats['targ_mean']


In [ ]:
# FUNCTIONS FOR INFERENCE

def inference_Lott(data):
    steps, lats, lons, levs = len(data), data[0]['ua'].shape[0], data[0]['ua'].shape[1], data[0]['ua'].shape[2]

    pdu_sso = np.zeros((steps,lats,lons,levs))
    pdv_sso = np.zeros((steps,lats,lons,levs))

    for z in range(steps):
        pum1 = data[z]['ua']
        pvm1 = data[z]['va']
        ptm1 = data[z]['ta']

        zhgeo = data[z]['zh']

        R_spec = 287.0
        pmair = papm1 / ptm1 / R_spec

        pmea = data[z]['geop'] / 9.81
        pstd = data[z]['oro_std']
        psig = data[z]['oro_slope']
        pgam = data[z]['oro_anis']
        pthe = data[z]['oro_angle']

        ppic = pmea * 2
        pval = pmea / 2

        psftlf = data[z]['land_sea_mask']

        for i in range(lats):
            for j in range(lons):
                id = (i, j)
                pdu_sso[z,i,j,:], pdv_sso[z,i,j,:] = L97.calc_SSO(pdtime_1h, zhgeo[id], paphm1, papm1, pmair[id], ptm1[id], pum1[id], pvm1[id],
                                                                  pmea[id], pstd[id], psig[id], pgam[id], pthe[id], ppic[id], pval[id], psftlf[id], pcoriolis[i])

    return pdu_sso*pdtime_24h, pdv_sso*pdtime_24h



def inference_ERA5(data, orostd_threshold):
    steps, lats, lons, levs = len(data), data[0]['ua'].shape[0], data[0]['ua'].shape[1], data[0]['ua'].shape[2]
    MFx_ERA5 = np.zeros((steps,lats,lons,levs))
    MFy_ERA5 = np.zeros((steps,lats,lons,levs))

    pdu_era5 = np.zeros((steps,lats,lons,levs))
    pdv_era5 = np.zeros((steps,lats,lons,levs))

    for z in range(steps):
        
        MFx = data[z]['zf']
        MFy = data[z]['mf']

        geop = data[z]['zh']
        ta = data[z]['ta']

        MFx_ERA5[z,:,:,:] = MFx[:,:,:]
        MFy_ERA5[z,:,:,:] = MFy[:,:,:]

        for i in range(lats):
            for j in range(lons):
                id = (i, j)
            
                if data[z]['land_sea_mask'][id] != 0 and data[z]['oro_std'][id] >= orostd_threshold:
                   # pdu_era5[z,i,j,:]= drag_from_flux2(MFx[id], papm1)
                   # pdv_era5[z,i,j,:]= drag_from_flux2(MFy[id], papm1)
                    pdu_era5[z,i,j,:]= drag_from_flux(MFx[id], geop[id], papm1, ta[id])
                    pdv_era5[z,i,j,:]= drag_from_flux(MFy[id], geop[id], papm1, ta[id])


                else:
                    pdu_era5[z,i,j,:] = np.zeros(37)
                    pdv_era5[z,i,j,:] = np.zeros(37)
    
    return MFx_ERA5, MFy_ERA5, pdu_era5*pdtime_24h, pdv_era5*pdtime_24h



def inference_NN(model, data, orostd_threshold):
    steps, lats, lons, levs = len(data), data[0]['ua'].shape[0], data[0]['ua'].shape[1], data[0]['ua'].shape[2]

    MFx_NN = np.zeros((steps,lats,lons,levs))
    MFy_NN = np.zeros((steps,lats,lons,levs))

    pdu_nn = np.zeros((steps,lats,lons,levs))
    pdv_nn = np.zeros((steps,lats,lons,levs))

    for z in range(steps):
        nn_vec = np.concatenate([data[z]['ua'], data[z]['va'], data[z]['wa'], data[z]['ta'],
                         np.expand_dims(data[z]['geop'], axis=2), np.expand_dims(data[z]['oro_std'], axis=2),
                         np.expand_dims(data[z]['oro_anis'], axis=2), np.expand_dims(data[z]['oro_angle'], axis=2),
                         np.expand_dims(data[z]['oro_slope'], axis=2)], axis=2)
     
        geop = data[z]['zh']
        ta = data[z]['ta']
        
        cols_scaled = torch.tensor(scale_features(nn_vec).reshape(-1,153))
        pred=model(cols_scaled).reshape(lats,lons,74)

        for i in range(lats):
            for j in range(lons):
                id = (i, j)

                with torch.no_grad():
                    MFx_NN[z,i,j,:] = rescale_fluxes(pred[id])[:37]
                    MFy_NN[z,i,j,:] = rescale_fluxes(pred[id])[37:]
                
                if data[z]['land_sea_mask'][id] != 0 and data[z]['oro_std'][id] >= orostd_threshold:
                    #pdu_nn[z,i,j,:]= drag_from_flux2(MFx_NN[z,i,j,:], papm1)
                    #pdv_nn[z,i,j,:]= drag_from_flux2(MFy_NN[z,i,j,:], papm1)
                   pdu_nn[z,i,j,:]= drag_from_flux(MFx_NN[z,i,j,:], geop[i,j,:], papm1, ta[i,j,:])
                   pdv_nn[z,i,j,:]= drag_from_flux(MFy_NN[z,i,j,:], geop[i,j,:], papm1, ta[i,j,:])
    
    return MFx_NN, MFy_NN, pdu_nn*pdtime_24h, pdv_nn*pdtime_24h

In [ ]:
# FUNCTION FOR PLOTTING

def triple_plot_ZAs(data1, data2, data3, limit=1, typus = '', scheme = '', savetitle = '', cmap='cmaps.vik'):


    plt.rcParams.update({'font.size': 29})
  
    fs1 = 32
    fs2 = 29
    labelpad = 7
    pad=24


    lat = np.linspace(70,-70,50)
    height = papm1/100
    if typus == 'GWD':
        norm = Normalize(vmin=-limit, vmax=limit)
    elif typus == 'MF':
        norm = SymLogNorm(linthresh=1e-2, vmin=-limit, vmax=limit )


    fig = plt.figure(figsize=(30,13), dpi=150)

    ax0 = fig.add_axes([0.052,  0.08,  0.32, 0.85])
    ax1 = fig.add_axes([0.402,  0.08,  0.32, 0.85])
    ax2 = fig.add_axes([0.752,  0.08,  0.32, 0.85])


    im = ax0.pcolormesh(lat, height, data1.T, cmap=cmap, norm=norm)#, vmin=-0.003, vmax=0.003)
    ax1.pcolormesh(lat, height, data2.T, cmap=cmap, norm=norm)#, vmin=-0.003, vmax=0.003)
    ax2.pcolormesh(lat, height, data3.T, cmap=cmap, norm=norm)#, vmin=-0.003, vmax=0.003)


    ax0.invert_yaxis()
    ax0.set_yscale('log')
    ax0.set_ylim([1000,1])

    ax1.invert_yaxis()
    ax1.set_yscale('log')
    ax1.set_ylim([1000,1])

    ax2.invert_yaxis()
    ax2.set_yscale('log')
    ax2.set_ylim([1000,1])

    ax0.set_xlabel('Latitude [deg]', fontsize=fs2, labelpad=labelpad)
    ax0.set_ylabel('Pressure [hPa]', fontsize=fs2, labelpad=0)
    ax1.set_xlabel('Latitude [deg]', fontsize=fs2, labelpad=labelpad)
    ax2.set_xlabel('Latitude [deg]', fontsize=fs2, labelpad=labelpad)
    ax1.tick_params(left=False, labelleft=False)
    ax2.tick_params(left=False, labelleft=False)


    formatter = FuncFormatter(lambda y, _: f'{int(y)}')
    ax0.yaxis.set_major_formatter(formatter)


    ax0.set_title('ERA5 ' + scheme + ' $' + typus + '_x$', fontsize=fs1, pad=pad)
    ax1.set_title('U-Net ' + scheme + ' $' + typus + '_x$', fontsize=fs1, pad=pad)
    ax2.set_title('Lott 97 ' + scheme + ' $' + typus + '_x$', fontsize=fs1, pad=pad)


    ax0.tick_params(axis='both', which='major', labelsize=fs2)
    ax1.tick_params(axis='both', which='major', labelsize=fs2)
    ax2.tick_params(axis='both', which='major', labelsize=fs2)


    if typus == 'GWD':
        custom_ticks = [-limit, -limit*.75, -limit*.5, -limit*.25,  0, limit*.25, limit*.5, limit*.75, limit]
        cbar = fig.colorbar(im, ax=[ax0,ax1,ax2], ticks=custom_ticks, pad=0.02, extend='both')#.035)
        cbar.set_label(label='m/s/day', fontsize=fs2, labelpad=0)
        cbar.ax.tick_params(labelsize=fs2) # Adjust color bar tick label font size

    # if typus == 'MF':
    #     custom_ticks = [-limit, -limit/10, -limit/100, -limit/1000, -limit/10000,  0, limit/10000, limit/1000, limit/100, limit/10, limit]
    #     cbar= fig.colorbar(im, ax=[ax[0],ax[1]], ticks=custom_ticks, pad=0.035)
    #     cbar.set_label(label='mPa', fontsize=fs2)
    #     cbar.ax.tick_params(labelsize=fs2) # Adjust color bar tick label font size

    plt.savefig('/p/project1/icon-a-ml/haslauer1/____project1/_publication_4500/_20260609_Lott-07.png', dpi=150)
    plt.show()

In [ ]:
# COMPUTE GWD WITH LOTT'S SCHEME

start_lott = time.time()
GWDx_Lott, GWDy_Lott = inference_Lott(data)
end_lott = time.time()
print('Inference time Lott: ')
print(end_lott-start_lott)

GWDx_Lott_ZA, GWDy_Lott_ZA = GWDx_Lott.mean(axis=2).mean(axis=0), GWDy_Lott.mean(axis=2).mean(axis=0)
GWDx_Lott_ZA.min(), GWDx_Lott_ZA.max(), GWDy_Lott_ZA.min(), GWDy_Lott_ZA.max()

In [ ]:
GWDx_Lott.shape

In [ ]:
# COMPUTE GWD IN ERA5

orostd_threshold = np.partition(data[0]['oro_std'].flatten(), 5003)[5003]
MFx_ERA5, MFy_ERA5, GWDx_ERA5, GWDy_ERA5 = inference_ERA5(data, orostd_threshold)
GWDx_ERA5_ZA, GWDy_ERA5_ZA = GWDx_ERA5.mean(axis=2).mean(axis=0), GWDy_ERA5.mean(axis=2).mean(axis=0)
GWDx_ERA5_ZA.min(), GWDx_ERA5_ZA.max(), GWDy_ERA5_ZA.min(), GWDy_ERA5_ZA.max()

In [ ]:
# COMPUTE GWD WITH NNs

model = torch.load('/p/scratch/icon-a-ml/haslauer1/_data/_ERA5/_ML_new/_n32/SGIG16/_preprocessed_full2024_land_orostd_/__models/UNet_2_2026-04-15_UNet_2_SG_orog.pt',
                   map_location=torch.device('cpu'), weights_only=False)

orostd_threshold = np.partition(data[0]['oro_std'].flatten(), 5003)[5003]

start_nn = time.time()
MFx_NN, MFy_NN, GWDx_NN, GWDy_NN = inference_NN(model, data, orostd_threshold)
end_nn = time.time()
print('Inference time NN: ')
print(end_nn-start_nn)

GWDx_NN_ZA, GWDy_NN_ZA = GWDx_NN.mean(axis=2).mean(axis=0), GWDy_NN.mean(axis=2).mean(axis=0)

In [ ]:
GWDx_NN_ZA.T.shape

In [ ]:
A = np.arange(16).reshape(4,4)
A, A.T

In [ ]:
# CREATE PLOT

triple_plot_ZAs(GWDx_ERA5_ZA, GWDx_NN_ZA, GWDx_Lott_ZA, limit=0.4, typus='GWD', scheme='2022-07-01', savetitle='', cmap=cmaps.vik)

In [ ]:
# CALCULATE R^2 VALUES

from sklearn.metrics import r2_score

print('R2 scores NN, Lott: ')
print(r2_score(GWDx_ERA5_ZA, GWDx_NN_ZA), r2_score(GWDx_ERA5_ZA, GWDx_Lott_ZA))